In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import xmltodict
import rasterio
import os
from pathlib import Path
from dateutil.parser import isoparse
from scipy.interpolate import CubicHermiteSpline

In [ ]:
data_dir = "/data/S1"
product_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20210624T172451_20210624T172518_038485_048A9C_A58F.SAFE"

In [ ]:
pth_tiff = Path(f"{product_dir}/measurement/").glob("*iw1*vv*.tiff")
pth_xml = Path(f"{product_dir}/annotation/").glob("*iw1*vv*.xml")
pth_tiff=list(pth_tiff)[0]
pth_xml=list(pth_xml)[0]

In [ ]:
with open(pth_xml) as f:
    meta = xmltodict.parse(f.read())

from rasterio.warp import calculate_default_transform, reproject

with rasterio.open(pth_tiff) as src:
    im = np.log(np.abs(src.read(1))+1)
    prof = src.profile.copy()
    # uncomment to reproject from GCPs
    # gcps, src_crs = src.gcps
    # dst_crs = "EPSG:4326"
    # dst_trans, dst_width, dst_height = calculate_default_transform(
    #     src_crs=src.crs,
    #     dst_crs="EPSG:4326",
    #     width=src.width,
    #     height=src.height,
    #     gcps=src.gcps[0],
    # )
    # prof.update(
    #     dict(width=dst_width,
    #     height=dst_height,
    #     dtype="float32",
    #     crs="EPSG:4326",
    #     transform=dst_trans,
    #     nodata=0,
    #     driver="COG",
    #     compress="deflate",
    #     resampling="bilinear"
    #     )
    # )
    # with rasterio.open(f"/data/res/toto.tif", "w", **prof) as dst:
    #     reproject(
    #         im,
    #         rasterio.band(dst, 1),
    #         src_crs=src_crs,
    #         dst_crs=dst_crs,
    #         # src_transform=src.transform,
    #         gcps=gcps,
    #         dst_transform=dst_trans,
    #     )
    #     print(dst.profile)

In [ ]:
# show first bursts
plt.figure(figsize=(15,10))
vmin, vmax = np.percentile(im[im>0][::16], [10,90])
plt.imshow(im[:3000][:,::4], vmin=vmin, vmax=vmax, cmap='grey')

In [ ]:
# from eo_tools.util import show_cog
# show_cog("/data/res/toto.tif", rescale="2,9")

In [ ]:
# general info
image_info = meta["product"]["imageAnnotation"]["imageInformation"]
azimuth_time_interval = image_info["azimuthTimeInterval"]
slant_range_time = image_info["slantRangeTime"]
range_pixel_spacing = image_info["rangePixelSpacing"]
product_info = meta["product"]["generalAnnotation"]["productInformation"]
range_sampling_rate = product_info["rangeSamplingRate"]
radar_frequency = product_info["radarFrequency"]

In [ ]:
# look for burst info
burst_info = meta["product"]["swathTiming"]
lines_per_burst = burst_info["linesPerBurst"]
samples_per_burst = burst_info["samplesPerBurst"]
burst_count = burst_info["burstList"]["@count"]
first_burst = burst_info["burstList"]["burst"][0]
az_time = first_burst["azimuthTime"]
az_anx_time = first_burst["azimuthAnxTime"]

In [ ]:
print(state_vectors)

In [ ]:
# state vectors
orbit_list = meta["product"]["generalAnnotation"]["orbitList"]
orbit_count = orbit_list["@count"]
state_vectors = orbit_list["orbit"]

tt0 = isoparse(state_vectors[0]["time"])
tt = [(isoparse(it["time"]) - tt0).total_seconds() for it in state_vectors]
# tt = [isoparse(it["time"])-isoparse(az_time) for it in state_vectors]
xx = [float(it["position"]["x"]) for it in state_vectors]
yy = [float(it["position"]["y"]) for it in state_vectors]
zz = [float(it["position"]["z"]) for it in state_vectors]

vxx = [float(it["velocity"]["x"]) for it in state_vectors]
vyy = [float(it["velocity"]["y"]) for it in state_vectors]
vzz = [float(it["velocity"]["z"]) for it in state_vectors]

interp_orb = CubicHermiteSpline(
    tt, np.array([xx, yy, zz]).T, np.array([vxx, vyy, vzz]).T
)
interp_orb_v = interp_orb.derivative(1)

In [ ]:
# interpolation at azimuth times

In [ ]:
# dem download and conversion to ECEF
# found a bounding box based on gcps